## Read the Bronze product table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_prd = spark.table(
    "e2e_project.bronze.crm_prd_info"
)

display(bronze_prd)

In [0]:
bronze_prd.printSchema()

print("Rows:", bronze_prd.count())

## Check duplicate product IDs

In [0]:
display(
    bronze_prd
    .groupBy("prd_id")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    bronze_prd
    .groupBy("prd_key")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

## Inspect product cost

In [0]:
display(
    bronze_prd
    .filter(
        F.col("prd_cost").isNull() |
        (F.col("prd_cost") < 0)
    )
)

For Silver, a missing product cost in this project can be standardized to 0.

In [0]:
F.coalesce(F.col("prd_cost"), F.lit(0))

prd_cost

NULL → 0

otherwise keep original value

## Inspect prd_line

In [0]:
display(
    bronze_prd
    .groupBy("prd_line")
    .count()
)

We'll convert them to descriptive labels:

M → Mountain

R → Road

S → Other Sales

T → Touring

NULL/other → n/a

## Understand the strange prd_key

In [0]:
display(
    bronze_prd.select(
        "prd_id",
        "prd_key",
        "prd_nm"
    )
)

AC_HE-HL-U509-R

│────│ └────────┘

category   product key

cat_id   = AC_HE

prd_key  = HL-U509-R

Why?

Because later the ERP product-category table contains a category identifier, and we need a common key to join CRM products with ERP categories.

## Create the category ID

We'll transform the first part of the key.

In [0]:
silver_prd = bronze_prd.withColumn(
    "cat_id",
    F.regexp_replace(
        F.substring(F.col("prd_key"), 1, 5),
        "-",
        "_"
    )
)

## Extract the actual product key

replace the original combined key with the product-specific part:

In [0]:
silver_prd = silver_prd.withColumn(
    "prd_key",
    F.substring(
        F.col("prd_key"),
        7,
        100
    )
)

We're separating two business concepts that were encoded inside one source field.

## Handle missing product cost

In [0]:
silver_prd = silver_prd.withColumn(
    "prd_cost",
    F.coalesce(
        F.col("prd_cost"),
        F.lit(0)
    )
)

## Standardize product line

In [0]:
silver_prd = silver_prd.withColumn(
    "prd_line",
    F.when(
        F.upper(F.trim(F.col("prd_line"))) == "M",
        "Mountain"
    )
    .when(
        F.upper(F.trim(F.col("prd_line"))) == "R",
        "Road"
    )
    .when(
        F.upper(F.trim(F.col("prd_line"))) == "S",
        "Other Sales"
    )
    .when(
        F.upper(F.trim(F.col("prd_line"))) == "T",
        "Touring"
    )
    .otherwise("n/a")
)

In [0]:
display(
    silver_prd
    .groupBy("prd_line")
    .count()
)

## Cast prd_start_dt to a real date

In [0]:
silver_prd = silver_prd.withColumn(
    "prd_start_dt",
    F.to_date(F.col("prd_start_dt"))
)

For each product

       ↓

sort its historical records
by start date

## fix prd_end_dt

In [0]:
product_window = (
    Window
    .partitionBy("prd_key")
    .orderBy("prd_start_dt")
)

In [0]:
silver_prd = silver_prd.withColumn(
    "_next_start_dt",
    F.lead("prd_start_dt").over(product_window)
)

In [0]:
silver_prd = silver_prd.withColumn(
    "prd_end_dt",
    F.date_sub(
        F.col("_next_start_dt"),
        1
    )
)

In [0]:
silver_prd = silver_prd.drop("_next_start_dt")

## Inspect your complete Silver product data

In [0]:
display(silver_prd)

In [0]:
silver_prd.printSchema()

## Validate dates 

In [0]:
display(
    silver_prd.filter(
        F.col("prd_end_dt") < F.col("prd_start_dt")
    )
)

In [0]:
display(
    silver_prd
    .select(
        "prd_key",
        "prd_nm",
        "prd_start_dt",
        "prd_end_dt"
    )
    .orderBy(
        "prd_key",
        "prd_start_dt"
    )
)

## Validate product line and costs

In [0]:
display(
    silver_prd
    .groupBy("prd_line")
    .count()
)

In [0]:
display(
    silver_prd.filter(
        F.col("prd_cost").isNull()
    )
)

## Select columns in a clean order

In [0]:
silver_prd = silver_prd.select(
    "prd_id",
    "cat_id",
    "prd_key",
    "prd_nm",
    "prd_cost",
    "prd_line",
    "prd_start_dt",
    "prd_end_dt"
)

display(silver_prd)

In [0]:
(
    silver_prd.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.crm_prd_info"
    )
)

## Write it to Silver

In [0]:
(
    silver_prd.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.crm_prd_info"
    )
)

                 Bronze product


                 prd_key

                    │
           ┌────────┴─────────┐
           ↓                  ↓
        cat_id             prd_key
           │                  │
           └────────┬─────────┘
                    ↓

             standardization

                    ↓

          cost NULL → 0

          line codes → labels

          dates → DATE

                    ↓

              Window function

                    ↓

      LEAD(next product start date)

                    ↓

        derive product end date

                    ↓
                    
             Silver product

Encounter window functions constantly in serious data engineering.